# A fixed development sample for claim interpretation

This notebook implements `docs/SEMANTIC_DEVELOPMENT_SAMPLE_PROTOCOL.md`: up to three documents from each nonempty cue combination, excluding the original three pilot documents. This is purposive stratified development sampling, not an independent test or a representative sample. Existing human labels and model outputs are not changed. No new semantic labels are suggested.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next((p for p in [current, *current.parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
base = ROOT / "data/processed"
required = ["semantic_candidate_screen_v1.csv", "semantic_screen_manifest_v1.json"]
if any(not (base / name).exists() for name in required):
    raise FileNotFoundError("Run notebook 06 first.")
manifest = json.loads((base / "semantic_screen_manifest_v1.json").read_text())
source = base / "model_corpus_v1.csv"
if hashlib.sha256(source.read_bytes()).hexdigest() != manifest["source_sha256"]:
    raise ValueError("Frozen corpus hash mismatch.")
module_path = ROOT / "src/semantic_screening.py"
if hashlib.sha256(module_path.read_bytes()).hexdigest() != manifest["screening_module_sha256"]:
    raise ValueError("Screening module changed; preserve prior results and version the screen.")
sys.path.insert(0, str(ROOT / "src"))
from semantic_screening import screen_documents, PATTERNS
if PATTERNS != manifest["patterns"]:
    raise ValueError("Regex rules differ from the saved screen.")
corpus = pd.read_csv(source, keep_default_na=False)
prepared = corpus.loc[corpus["preparation_status"].eq("ready_for_exploration")]
expected = screen_documents(prepared)
screened = pd.read_csv(base / "semantic_candidate_screen_v1.csv", keep_default_na=False)
screened = screened.sort_values("execution_number").reset_index(drop=True)
pd.testing.assert_frame_equal(screened, expected, check_dtype=False)
print("Verified 479-document screen against frozen source and recorded rules.")


In [ ]:
SEED = 20261106
PILOT_IDS = {117, 391, 499}
POOL = screened.loc[~screened["execution_number"].isin(PILOT_IDS)].copy()
rng = np.random.default_rng(SEED)
parts, allocation = [], []
for combination, frame in POOL.groupby("cue_combination", sort=True):
    frame = frame.sort_values("execution_number").reset_index(drop=True)
    n = min(3, len(frame))
    chosen = frame.iloc[rng.choice(len(frame), size=n, replace=False)].copy()
    parts.append(chosen)
    allocation.append({"cue_combination": combination,
                       "eligible_documents": len(frame), "selected_documents": n})
sample = pd.concat(parts, ignore_index=True).sort_values(
    ["candidate_document", "cue_combination", "execution_number"],
    ascending=[False, True, True],
).reset_index(drop=True)
old_annotations = pd.read_csv(base / "annotation_development_v2.csv", keep_default_na=False)
sample["in_old_development_sample"] = sample["execution_number"].isin(old_annotations["execution_number"])
sample["exposure_status"] = np.where(
    sample["in_old_development_sample"], "known_previous_development_review",
    "other_prior_exposure_not_established",
)
sample["selection_use"] = "development_only_not_independent_evaluation"
sample["source_sha256"] = manifest["source_sha256"]
sample["sampling_seed"] = SEED
assert sample["execution_number"].is_unique
assert not sample["execution_number"].isin(PILOT_IDS).any()
assert len(sample) == 21 and int(sample["candidate_document"].sum()) == 15
published_selection = json.loads((ROOT / "reports/semantic_development_sampling_v1.json").read_text())
if sample["execution_number"].astype(int).tolist() != published_selection["execution_ids_in_display_order"]:
    raise ValueError("Sample differs from the recorded selection; investigate environment or input changes before review.")
print("Selected: 21 documents; 15 cue candidates and six nonmatches.")
display(pd.DataFrame(allocation).set_index("cue_combination"))
print("All selected documents remain development material regardless of exposure status.")


## Preserve selection separately from working notes

The selected source table and sampling manifest are immutable per version. The separate worksheet starts with pending review status and empty evidence/note fields. If it exists already, source fields must match; notes are retained. A document with no useful target claim is a valid reviewed observation and is not replaced.


In [ ]:
def save_new_or_identical(path, content):
    if path.exists():
        if path.read_bytes() != content:
            raise FileExistsError(f"Existing artifact differs; use a new version: {path.name}")
        print("Verified existing:", path.relative_to(ROOT))
    else:
        with path.open("xb") as handle:
            handle.write(content)
        print("Created:", path.relative_to(ROOT))

selection_path = base / "semantic_development_sample_v1.csv"
save_new_or_identical(selection_path, sample.to_csv(index=False).encode("utf-8-sig"))
worksheet_path = base / "semantic_development_review_v1.csv"
source_columns = sample.columns.tolist()
review_columns = ["review_status", "evidence_span", "review_note"]
if worksheet_path.exists():
    review = pd.read_csv(worksheet_path, keep_default_na=False)
    if set(review.columns) != set(source_columns + review_columns) or not review["execution_number"].is_unique:
        raise ValueError("Existing review worksheet schema or IDs differ.")
    if set(review["execution_number"]) != set(sample["execution_number"]):
        raise ValueError("Existing worksheet contains different documents.")
    review = review.set_index("execution_number").loc[sample["execution_number"]].reset_index()
    pd.testing.assert_frame_equal(review[source_columns], sample[source_columns], check_dtype=False)
    print("Existing review worksheet retained; no notes overwritten.")
else:
    review = sample.copy()
    review["review_status"] = "pending"
    review["evidence_span"] = ""
    review["review_note"] = ""
    save_new_or_identical(worksheet_path, review.to_csv(index=False).encode("utf-8-sig"))
selection_manifest = {
    "sampling_seed": SEED,
    "rng": "numpy.random.default_rng; sorted combinations and execution IDs",
    "numpy": np.__version__, "pandas": pd.__version__,
    "source_sha256": manifest["source_sha256"],
    "screening_table_sha256": hashlib.sha256((base / "semantic_candidate_screen_v1.csv").read_bytes()).hexdigest(),
    "annotation_source_sha256": hashlib.sha256((base / "annotation_development_v2.csv").read_bytes()).hexdigest(),
    "excluded_pilot_ids": sorted(PILOT_IDS),
    "execution_ids_in_display_order": sample["execution_number"].astype(int).tolist(),
    "allocation": allocation,
    "scope": "Development review only; no unseen status, labels or performance inferred",
}
save_new_or_identical(base / "semantic_development_sample_manifest_v1.json",
                     (json.dumps(selection_manifest, indent=2) + "\n").encode())


## Begin reading the fixed sample

The first two documents below follow the declared display order. Read the complete prepared text before selecting any clause. Cue hits do not establish the source, event actor or polarity of a proposition. For now discuss possible spans and ambiguity; do not automatically apply the four fields to a whole document. To continue later, call `show_review_documents(start=2, n=2)` without redrawing the sample.


In [ ]:
def show_review_documents(start=0, n=2):
    if start < 0 or n < 1:
        raise ValueError("Use a nonnegative start and a positive count.")
    for index, row in sample.iloc[start:start+n].iterrows():
        print("\n" + "=" * 70)
        print(f"Review position {index + 1} / {len(sample)}; execution {int(row['execution_number'])}")
        print("Cue group:", row["cue_combination"])
        print("Exposure:", row["exposure_status"])
        print("Source:", row["source_url"])
        print(row["model_text"])

show_review_documents(start=0, n=2)
